# Operatic Singer Cohort — Acoustic Feature Analysis

**Scope:** Audio-only analysis of the 2 pre-cohort lyrical/operatic singer recordings.  
**Goal:** Exhaustive feature extraction with relevancy notes; inform the final feature set for the full cohort.  
**Pipeline base:** `pneumophonic_analysis` package (M1/M2 framework).  
**New features:** `singer_acoustc_features.py` (singer's formant ratio, LTAS slope, vibrato, band HNR).

---
## How to use
1. Set `DATA_ROOT` and `SUBJECTS` in the **Config** cell.
2. Run all cells sequentially.
3. Each feature section has a `# RELEVANCE:` comment explaining why (or why not) the feature matters for the operatic study.

Task labels follow `OPERATIC_COHORT_PROTOCOL.md`.

## 0. Imports & Config

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
import parselmouth
from parselmouth.praat import call as praat_call
from scipy import signal as sp_signal
from scipy.stats import linregress, pearsonr

import noisereduce as nr #use wiener filter instead

In [2]:


warnings.filterwarnings('ignore')

# Add repo root to path so the package is importable without install
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pneumophonic_analysis.config import get_config
from pneumophonic_analysis.audio_processing import AudioProcessor
from pneumophonic_analysis.acoustic_features import (
    extract_pitch_metrics, extract_perturbation_metrics,
    extract_formant_metrics, extract_voice_quality_metrics
)
from pneumophonic_analysis.singer_acoustc_features import (
    compute_singer_features, compute_singers_formant_ratio,
    compute_ltas_slope, compute_vibrato, compute_formant_band_hnr
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

CFG = get_config()

ImportError: cannot import name 'extract_pitch_metrics' from 'pneumophonic_analysis.acoustic_features' (C:\Users\Matéo\OneDrive\Documents\GitHub\pneumophonic_pipeline\pneumophonic_analysis\acoustic_features.py)

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
# Point DATA_ROOT at the singer data folder.
# Expected structure:
#   DATA_ROOT/
#     <SubjectID>/
#       renders/
#         vowel_ref.wav
#         scale5_t1.wav  ...  scale5_t8.wav
#         scale3_t1.wav  ...
#         scale_asc_desc.wav
#         scarb_girata.wav
#         scarb_non_girata.wav
#         aria_t1.wav  aria_t2.wav  aria_t3.wav

DATA_ROOT = Path(r'C:\Users\Matéo\OneDrive\Documents\GitHub\pneumophonic_pipeline\data_root\singer_subjects')

# Pre-cohort subjects — adjust to actual folder names
SUBJECTS = [
    {'id': 'Subject01', 'voice_type': 'mezzo_soprano', 'sex': 'F', 'age': None},
    {'id': 'Subject02', 'voice_type': 'soprano',       'sex': 'F', 'age': None},
]

SR_TARGET = 48000   # All audio resampled to this rate
HOP      = 720      # pYIN / STFT hop (≈15 ms at 48 kHz → ~66.67 fps)
FRAME    = 1440     # STFT frame length (30 ms)
F0_FRAME_RATE = SR_TARGET / HOP   # ~66.67 Hz

# ─────────────────────────────────────────────────────────────────────────────
print(f"Data root : {DATA_ROOT}")
print(f"Subjects  : {[s['id'] for s in SUBJECTS]}")
print(f"SR={SR_TARGET} Hz | hop={HOP} | frame={FRAME} | F0_fps={F0_FRAME_RATE:.2f}")

## 1. Audio Loading & Preprocessing

In [ ]:
# RELEVANCE: Preprocessing is identical to the healthy cohort (Zocco protocol).
# Stationary noise reduction is conservative (prop_decrease=0.85) to avoid
# artefacts on the harmonic-rich singing signal. Pre-emphasis boosts highs
# before STFT-based analysis. For vibrato and LTAS, we work on the
# noise-reduced but NOT pre-emphasised signal to preserve spectral shape.

def load_audio(wav_path: Path, sr: int = SR_TARGET) -> tuple[np.ndarray, int]:
    """Load, resample to `sr`, and apply stationary noise reduction."""
    audio, orig_sr = librosa.load(str(wav_path), sr=None, mono=True)
    if orig_sr != sr:
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=sr)
    audio_clean = nr.reduce_noise(
        y=audio, sr=sr,
        stationary=True,
        prop_decrease=0.85
    )
    return audio_clean, sr


def preemphasis(audio: np.ndarray, coef: float = 0.97) -> np.ndarray:
    return np.append(audio[0], audio[1:] - coef * audio[:-1])


# Build a {subject_id: {task_label: audio_array}} dict for every found file
audio_data: dict[str, dict[str, np.ndarray]] = {}

for subj in SUBJECTS:
    sid = subj['id']
    render_dir = DATA_ROOT / sid / 'renders'
    audio_data[sid] = {}
    if not render_dir.exists():
        print(f"[WARN] {render_dir} not found — skipping {sid}")
        continue
    for wav in sorted(render_dir.glob('*.wav')):
        task = wav.stem
        try:
            aud, _ = load_audio(wav)
            audio_data[sid][task] = aud
            print(f"  {sid}/{task}: {len(aud)/SR_TARGET:.2f}s")
        except Exception as e:
            print(f"  [ERR] {sid}/{task}: {e}")

print("\nLoaded tasks per subject:")
for sid, tasks in audio_data.items():
    print(f"  {sid}: {list(tasks.keys())}")

## 2. F0 Extraction (pYIN)

In [ ]:
# RELEVANCE: F0 is the single most important feature for operatic analysis.
# It underlies vibrato, pitch accuracy across scale tasks, tessitura coverage,
# and the voce girata / non-girata comparison (girata tends to produce a
# slightly higher perceived pitch centre and more stable vibrato).
#
# pYIN (probabilistic YIN) is chosen over SHS or CREPE because:
#   - handles the wide F0 range of trained singers (soprano up to ~1100 Hz)
#   - produces voiced probability which gates vibrato extraction
#   - consistent with Zocco protocol for FRC analysis comparability
#
# F0 range expanded to 50–1200 Hz (vs 50–500 in healthy cohort) to cover
# soprano high C (C6 ≈ 1047 Hz) and above.

F0_MIN = 50    # Hz — covers bass; below vocal fry regime
F0_MAX = 1200  # Hz — covers soprano C6 and occasional high notes

f0_data: dict[str, dict[str, np.ndarray]] = {}  # {sid: {task: f0_hz_array}}
voiced_data: dict[str, dict[str, np.ndarray]] = {}

for sid, tasks in audio_data.items():
    f0_data[sid] = {}
    voiced_data[sid] = {}
    for task, audio in tasks.items():
        f0, voiced_flag, voiced_probs = librosa.pyin(
            audio,
            fmin=F0_MIN, fmax=F0_MAX,
            sr=SR_TARGET,
            hop_length=HOP,
            n_thresholds=3,
        )
        # Mask unvoiced frames with NaN
        f0_masked = np.where(voiced_flag, f0, np.nan)
        f0_data[sid][task] = f0_masked
        voiced_data[sid][task] = voiced_flag

# Quick visualisation — F0 contour for every task
for sid, tasks in f0_data.items():
    n = len(tasks)
    if n == 0:
        continue
    fig, axes = plt.subplots(n, 1, figsize=(14, 2.2 * n), sharex=False)
    if n == 1:
        axes = [axes]
    for ax, (task, f0) in zip(axes, tasks.items()):
        t = np.arange(len(f0)) / F0_FRAME_RATE
        ax.plot(t, f0, lw=0.8, color='steelblue')
        ax.set_ylabel('F0 (Hz)', fontsize=8)
        ax.set_title(f'{sid} — {task}', fontsize=9)
        ax.set_ylim(0, F0_MAX)
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'pYIN F0 contours — {sid}', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. F0 Summary Statistics

In [ ]:
# RELEVANCE: Per-task mean/median/SD/range of F0.
# Median F0 is more robust than mean for singing (wide range, unvoiced gaps).
# F0 range across tasks maps the tessitura actually exploited.
# F0 SD captures pitch variability — high in scales, low in sustained vowels.
# Useful as a covariate to control pitch in OEP correlation models.

rows = []
for sid, tasks in f0_data.items():
    for task, f0 in tasks.items():
        voiced_f0 = f0[~np.isnan(f0)]
        if len(voiced_f0) == 0:
            continue
        rows.append({
            'subject': sid,
            'task': task,
            'f0_mean_hz':   round(np.mean(voiced_f0), 1),
            'f0_median_hz': round(np.median(voiced_f0), 1),
            'f0_sd_hz':     round(np.std(voiced_f0, ddof=1), 1),
            'f0_min_hz':    round(np.min(voiced_f0), 1),
            'f0_max_hz':    round(np.max(voiced_f0), 1),
            'f0_range_hz':  round(np.ptp(voiced_f0), 1),
            'voiced_frac':  round(np.mean(~np.isnan(f0)), 3),
            'duration_s':   round(len(f0) / F0_FRAME_RATE, 2),
        })

df_f0 = pd.DataFrame(rows)
display(df_f0)

## 4. Vibrato (Sundberg 2012)

In [ ]:
# RELEVANCE: Vibrato is the defining feature of the classical/operatic singing
# style and a *primary OEP-correlation candidate*.
#
# Mechanism: vibrato is driven by rhythmic modulation of subglottal pressure
# (respiratory system). Thus:
#   - vibrato rate should correlate with respiratory oscillation frequency
#   - vibrato extent should correlate with OEP flow amplitude
#   - voce girata is expected to produce more regular, deeper vibrato than
#     voce non-girata (hypothesis to test)
#
# Three outputs from compute_vibrato():
#   rate_hz      — should be 5–6 Hz in Western classical; 3.5 Hz in Peking opera
#   extent_cents — peak-to-peak; 50–150 cents is normal for trained singers
#   regularity   — spectral peak prominence (0–1); low = tremolo or irregular
#
# NOTE: vibrato is unreliable on short segments (<1 s voiced) and on the
# ascending/descending scale tasks (rapid pitch changes mask the modulation).
# Filter by regularity > 0.3 before interpreting rate/extent.

rows_vib = []
for sid, tasks in f0_data.items():
    for task, f0 in tasks.items():
        rate, extent, reg = compute_vibrato(f0, F0_FRAME_RATE)
        rows_vib.append({
            'subject': sid,
            'task': task,
            'vibrato_rate_hz':      round(rate, 2) if not np.isnan(rate) else np.nan,
            'vibrato_extent_cents': round(extent, 1) if not np.isnan(extent) else np.nan,
            'vibrato_regularity':   round(reg, 3) if not np.isnan(reg) else np.nan,
        })

df_vib = pd.DataFrame(rows_vib)
display(df_vib)

# Box plots: vibrato rate and extent — girata vs non-girata vs aria
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, ylabel in zip(
    axes,
    ['vibrato_rate_hz', 'vibrato_extent_cents', 'vibrato_regularity'],
    ['Rate (Hz)', 'Extent (cents PP)', 'Regularity (0–1)']
):
    sub = df_vib.dropna(subset=[col])
    if sub.empty:
        ax.set_title(f'{col}\n(no data yet)')
        continue
    sns.stripplot(data=sub, x='task', y=col, hue='subject', ax=ax, dodge=True)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    ax.set_title(col.replace('_', ' ').title())
fig.suptitle('Vibrato parameters per task', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Singer's Formant Cluster (Cabrera 2011)

In [ ]:
# RELEVANCE: The singer's formant (2–4 kHz cluster) is the acoustic signature
# of operatic vocal training. Its energy ratio measures how much energy a
# singer concentrates in the projection band relative to the full spectrum.
#
# Expected: higher ratio in voce girata than non-girata; higher in aria than
# scale tasks. Sopranos at high pitch may not show a distinct cluster because
# harmonics thin above ~2 kHz.
#
# OEP connection: weakly expected to correlate with Vab (belly phonation
# increases sub-glottal pressure → brighter spectrum). Treat as secondary.

rows_sfr = []
for sid, tasks in audio_data.items():
    for task, audio in tasks.items():
        ratio, ratio_db = compute_singers_formant_ratio(audio, SR_TARGET)
        rows_sfr.append({
            'subject': sid, 'task': task,
            'sfr_linear': round(ratio, 4) if not np.isnan(ratio) else np.nan,
            'sfr_db':     round(ratio_db, 2) if not np.isnan(ratio_db) else np.nan,
        })

df_sfr = pd.DataFrame(rows_sfr)
display(df_sfr)

fig, ax = plt.subplots(figsize=(12, 4))
df_plot = df_sfr.dropna(subset=['sfr_db'])
if not df_plot.empty:
    sns.barplot(data=df_plot, x='task', y='sfr_db', hue='subject', ax=ax)
    ax.set_ylabel("Singer's formant ratio (dB)")
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    ax.axhline(-13, ls='--', lw=0.8, color='gray', label='~5% energy reference')
ax.set_title("Singer's Formant Cluster Energy Ratio — Cabrera 2011")
plt.tight_layout()
plt.show()

## 6. LTAS Slope (Sundberg 2012)

In [ ]:
# RELEVANCE: Long-term average spectrum slope (dB/octave, 700–6000 Hz) captures
# spectral tilt — the balance between low-frequency power and high-frequency
# content.
#
# Sundberg et al. 2012 found slope decreases ~0.2 dB/oct per dB SPL increase
# in Peking opera singers. For Western operatic singers the relationship is
# expected to be similar: louder phonation → shallower (less negative) slope.
#
# OEP connection: Vcw (total chest wall volume) correlates with loudness in
# Zocco's data, so LTAS slope may be a useful proxy for respiratory drive.
#
# Filter: discard estimates with R² < 0.5 (poor fit, usually in very short
# or heavily modulated segments like fast scales).

rows_ltas = []
for sid, tasks in audio_data.items():
    for task, audio in tasks.items():
        slope, r2 = compute_ltas_slope(audio, SR_TARGET)
        rows_ltas.append({
            'subject': sid, 'task': task,
            'ltas_slope':  round(slope, 2) if not np.isnan(slope) else np.nan,
            'ltas_r2':     round(r2, 3)   if not np.isnan(r2) else np.nan,
        })

df_ltas = pd.DataFrame(rows_ltas)
display(df_ltas)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, ylabel in zip(
    axes,
    ['ltas_slope', 'ltas_r2'],
    ['Slope (dB/octave)', 'R²']
):
    sub = df_ltas.dropna(subset=[col])
    if sub.empty:
        continue
    sns.stripplot(data=sub, x='task', y=col, hue='subject', ax=ax, dodge=True)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
fig.suptitle('LTAS Slope — Sundberg 2012', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Perturbation Metrics — Jitter & Shimmer

In [ ]:
# RELEVANCE: Jitter (cycle-to-cycle F0 perturbation) and shimmer
# (cycle-to-cycle amplitude perturbation) are classical voice quality
# indicators designed for SUSTAINED, STEADY phonation.
#
# For operatic singers they have LIMITED UTILITY on scale tasks and arias
# because:
#   - vibrato inherently introduces large F0 perturbation (jitter will be
#     artificially high; not pathological)
#   - rapid pitch transitions in scales break the period-tracking assumption
#
# Valid use: sustained vowel (vowel_ref) and the Scarborough Fair strofa
# during steady-state held notes. Compare girata vs non-girata: lower jitter
# in girata is expected (better respiratory support → more stable phonation).
#
# Flag: if voiced_frac < 0.7 or F0 SD > 30 Hz, mark the estimate as
# 'unreliable for perturbation' — too much pitch movement.

def praat_perturbation(audio: np.ndarray, sr: int) -> dict:
    """Extract jitter and shimmer via Praat parselmouth."""
    snd = parselmouth.Sound(audio, sampling_frequency=float(sr))
    pitch = snd.to_pitch_cc(time_step=0.0, pitch_floor=50.0, pitch_ceiling=1200.0)
    pointprocess = praat_call([snd, pitch], 'To PointProcess (cc)')

    def _safe(fn, *args, default=np.nan):
        try:
            return fn(*args)
        except Exception:
            return default

    pp_args = (0.0001, 0.02, 1.3)
    return {
        'jitter_local':    _safe(praat_call, pointprocess, 'Get jitter (local)', 0, 0, *pp_args),
        'jitter_rap':      _safe(praat_call, pointprocess, 'Get jitter (rap)',   0, 0, *pp_args),
        'shimmer_local':   _safe(praat_call, [snd, pointprocess], 'Get shimmer (local)',   0, 0, *pp_args, 1.6),
        'shimmer_local_db':_safe(praat_call, [snd, pointprocess], 'Get shimmer (local, dB)', 0, 0, *pp_args, 1.6),
        'shimmer_apq3':    _safe(praat_call, [snd, pointprocess], 'Get shimmer (apq3)',    0, 0, *pp_args, 1.6),
        'shimmer_apq11':   _safe(praat_call, [snd, pointprocess], 'Get shimmer (apq11)',   0, 0, *pp_args, 1.6),
    }

rows_pert = []
for sid, tasks in audio_data.items():
    for task, audio in tasks.items():
        pert = praat_perturbation(audio, SR_TARGET)
        # Determine reliability flag
        f0 = f0_data[sid].get(task, np.array([np.nan]))
        voiced_f0 = f0[~np.isnan(f0)]
        reliable = (len(voiced_f0) > 0
                    and np.mean(~np.isnan(f0)) >= 0.7
                    and np.std(voiced_f0, ddof=1) < 30)
        rows_pert.append({'subject': sid, 'task': task, 'reliable': reliable, **pert})

df_pert = pd.DataFrame(rows_pert)
# Highlight unreliable rows
display(df_pert.style.apply(
    lambda row: ['background: #ffe0e0'] * len(row) if not row['reliable'] else [''] * len(row),
    axis=1
))

## 8. HNR — Full Spectrum & Band-Limited (Ikuma 2025)

In [ ]:
# RELEVANCE: Harmonics-to-Noise Ratio measures the proportion of periodic
# (harmonic) vs aperiodic (noise) energy in the voice.
#
# Full-spectrum HNR is the classical measure; it is dominated by low-frequency
# energy and tends to be insensitive to differences in the upper spectrum.
#
# Band-limited HNR (Ikuma et al. J. Voice 2025):
#   - hnr_f1_band: 300–1000 Hz — captures F1 region, relates to vowel clarity
#   - hnr_f2_band: 800–2500 Hz — captures F2 region, relates to consonant clarity
#   - hnr_full:    80–5000 Hz  — whole-voice reference
#
# For operatic singers: high HNR across all bands is expected; voce girata
# should produce higher HNR than non-girata (better glottal adduction).
# Vibrato does NOT degrade HNR (unlike jitter) because vibrato is periodic.

rows_hnr = []
for sid, tasks in audio_data.items():
    for task, audio in tasks.items():
        hnr_f1, hnr_f2, hnr_full = compute_formant_band_hnr(audio, SR_TARGET)
        # Also compute standard Praat HNR on full signal
        snd = parselmouth.Sound(audio, sampling_frequency=float(SR_TARGET))
        try:
            harm = snd.to_harmonicity_cc(time_step=0.01, minimum_pitch=50.0)
            vals = harm.values[harm.values > -100]
            hnr_praat = float(np.mean(vals)) if vals.size else np.nan
        except Exception:
            hnr_praat = np.nan

        rows_hnr.append({
            'subject': sid, 'task': task,
            'hnr_praat_db': round(hnr_praat, 2),
            'hnr_f1_db':   round(hnr_f1, 2)   if not np.isnan(hnr_f1) else np.nan,
            'hnr_f2_db':   round(hnr_f2, 2)   if not np.isnan(hnr_f2) else np.nan,
            'hnr_full_db': round(hnr_full, 2)  if not np.isnan(hnr_full) else np.nan,
        })

df_hnr = pd.DataFrame(rows_hnr)
display(df_hnr)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
for ax, col in zip(axes, ['hnr_praat_db', 'hnr_f1_db', 'hnr_f2_db', 'hnr_full_db']):
    sub = df_hnr.dropna(subset=[col])
    if sub.empty:
        continue
    sns.boxplot(data=sub, x='subject', y=col, ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
fig.suptitle('HNR — Full vs Band-limited (Ikuma 2025)', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Formants (F1, F2, F3)

In [ ]:
# RELEVANCE: Formants define vowel identity and resonance strategy.
#
# For operatic singers:
#   - Sopranos at high pitch manipulate F1 to track harmonics (F1 ≈ F0 strategy)
#     → formants are UNRELIABLE above ~700 Hz F0 (soprano passaggio)
#   - F1/F2 during voce girata vs non-girata captures the resonance adjustment
#     that underpins the "covered" vowel technique (F1 lowering, F2 rounding)
#   - F3 contributes to the singer's formant cluster; high F3 in men
#
# Praat Burg method: max_formants=5, max_frequency=5500 Hz (raised from 5000
# in Zocco for soprano high-frequency formants), window=25 ms.
#
# Critical note: formant tracking on rapid scale runs (scale5, scale3) is
# very unstable. Treat formant data from scale tasks as exploratory only.

def praat_formants(audio: np.ndarray, sr: int,
                   max_freq: float = 5500.0) -> dict:
    snd = parselmouth.Sound(audio, sampling_frequency=float(sr))
    formant = praat_call(
        snd, 'To Formant (burg)',
        0.0025, 5, max_freq, 0.025, 50.0
    )
    n_frames = praat_call(formant, 'Get number of frames')
    f1_vals, f2_vals, f3_vals = [], [], []
    for i in range(1, n_frames + 1):
        for lst, fn in [(f1_vals, 1), (f2_vals, 2), (f3_vals, 3)]:
            try:
                v = praat_call(formant, 'Get value at time', fn,
                               praat_call(formant, 'Get time from frame number', i),
                               'Hertz', 'Linear')
                lst.append(v if v == v else np.nan)  # NaN propagation
            except Exception:
                lst.append(np.nan)
    for lst in [f1_vals, f2_vals, f3_vals]:
        arr = np.array(lst, dtype=float)
    return {
        'f1_mean':   np.nanmean(f1_vals), 'f1_median': np.nanmedian(f1_vals),
        'f2_mean':   np.nanmean(f2_vals), 'f2_median': np.nanmedian(f2_vals),
        'f3_mean':   np.nanmean(f3_vals), 'f3_median': np.nanmedian(f3_vals),
    }

rows_fmt = []
for sid, tasks in audio_data.items():
    for task, audio in tasks.items():
        fmt = praat_formants(audio, SR_TARGET)
        rows_fmt.append({'subject': sid, 'task': task, **{k: round(v, 1) for k, v in fmt.items()}})

df_fmt = pd.DataFrame(rows_fmt)
display(df_fmt)

# F1/F2 vowel plot per subject (only for vowel_ref and Scarborough tasks)
vowel_tasks = [t for t in df_fmt['task'].unique() if 'vowel' in t or 'scarb' in t]
fig, ax = plt.subplots(figsize=(7, 6))
for sid, grp in df_fmt[df_fmt['task'].isin(vowel_tasks)].groupby('subject'):
    ax.scatter(grp['f2_median'], grp['f1_median'], label=sid, s=80)
    for _, row in grp.iterrows():
        ax.annotate(row['task'], (row['f2_median'], row['f1_median']),
                    fontsize=7, alpha=0.7)
ax.invert_xaxis(); ax.invert_yaxis()
ax.set_xlabel('F2 (Hz)'); ax.set_ylabel('F1 (Hz)')
ax.set_title('F1/F2 vowel space — sustained & Scarborough tasks')
ax.legend()
plt.tight_layout()
plt.show()

## 10. MFCCs

In [ ]:
# RELEVANCE: MFCCs encode spectral shape compactly (13 coefficients after
# mel-warping and DCT). They were shown in the healthy cohort (M2) to carry
# significant correlations with OEP respiratory compartments.
#
# For the operatic cohort:
#   - MFCC-1 (after C0) tracks overall spectral level — correlates with RMS
#     and likely with Vcw
#   - MFCC-2/3 track spectral tilt — may relate to LTAS slope (feature 6)
#   - Higher-order MFCCs capture fine spectral detail; less interpretable
#
# For scale tasks: per-frame MFCCs show the vowel trajectory as pitch rises.
# For girata/non-girata: mean MFCC profile should differ (covered vowel has
# different spectral envelope). Visualise as heatmap per task.
#
# Normalisation: z-score each coefficient across time within a segment
# (delta-normalisation) to remove DC offsets before comparison.

mfcc_data: dict[str, dict[str, np.ndarray]] = {}

for sid, tasks in audio_data.items():
    mfcc_data[sid] = {}
    for task, audio in tasks.items():
        audio_pe = np.append(audio[0], audio[1:] - 0.97 * audio[:-1])  # pre-emphasis
        mfcc = librosa.feature.mfcc(
            y=audio_pe, sr=SR_TARGET,
            n_mfcc=13,
            n_fft=FRAME, hop_length=HOP,
            n_mels=64
        )
        mfcc_data[sid][task] = mfcc  # shape: (13, n_frames)

# Heatmap: mean MFCC across time, one row per task
for sid, tasks in mfcc_data.items():
    mat = np.array([np.mean(m, axis=1) for m in tasks.values()])  # (n_tasks, 13)
    fig, ax = plt.subplots(figsize=(12, 0.5 * len(tasks) + 1))
    sns.heatmap(
        mat, ax=ax,
        xticklabels=[f'C{i}' for i in range(13)],
        yticklabels=list(tasks.keys()),
        cmap='RdBu_r', center=0, annot=True, fmt='.1f', annot_kws={'size': 7}
    )
    ax.set_title(f'Mean MFCCs per task — {sid}')
    ax.set_xlabel('MFCC coefficient')
    plt.tight_layout()
    plt.show()

## 11. Spectral Features — RMS, Centroid, Rolloff, Bandwidth

In [ ]:
# RELEVANCE:
#   RMS energy   — proxy for loudness / respiratory drive; strong OEP correlation
#                  candidate (Vcw ↑ → louder phonation)
#   Spec centroid— perceptual 'brightness'; increases with F0 and singer's formant
#                  enhancement; girata may show higher centroid
#   Spec rolloff — frequency below which 85% of energy lies; tracks spectral
#                  extension; less interpretable for singing than centroid
#   Spec bandwidth— spread around centroid; high in rich-harmonic operatic voice
#
# None of these capture the time course adequately as a single scalar.
# Report mean ± SD per task; use as covariates in the OEP correlation model.

rows_spec = []
for sid, tasks in audio_data.items():
    for task, audio in tasks.items():
        rms    = librosa.feature.rms(y=audio, frame_length=FRAME, hop_length=HOP)[0]
        cent   = librosa.feature.spectral_centroid(y=audio, sr=SR_TARGET,
                                                    n_fft=FRAME, hop_length=HOP)[0]
        roll   = librosa.feature.spectral_rolloff(y=audio, sr=SR_TARGET,
                                                   n_fft=FRAME, hop_length=HOP)[0]
        bw     = librosa.feature.spectral_bandwidth(y=audio, sr=SR_TARGET,
                                                     n_fft=FRAME, hop_length=HOP)[0]
        zcr    = librosa.feature.zero_crossing_rate(y=audio, frame_length=FRAME,
                                                     hop_length=HOP)[0]
        rows_spec.append({
            'subject': sid, 'task': task,
            'rms_mean': round(float(np.mean(rms)), 5),
            'rms_sd':   round(float(np.std(rms, ddof=1)), 5),
            'centroid_hz': round(float(np.mean(cent)), 1),
            'rolloff_hz':  round(float(np.mean(roll)), 1),
            'bandwidth_hz':round(float(np.mean(bw)), 1),
            'zcr_mean':    round(float(np.mean(zcr)), 5),
        })

df_spec = pd.DataFrame(rows_spec)
display(df_spec)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat,
                   ['rms_mean', 'rms_sd', 'centroid_hz', 'rolloff_hz', 'bandwidth_hz', 'zcr_mean']):
    sub = df_spec.dropna(subset=[col])
    if sub.empty:
        continue
    sns.barplot(data=sub, x='task', y=col, hue='subject', ax=ax)
    ax.set_title(col); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=45)
fig.suptitle('Spectral summary features per task', fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Mel Spectrogram — Time-Frequency Overview

In [ ]:
# RELEVANCE: Mel spectrograms are not used as summary statistics but as
# diagnostic visualisations and potential direct inputs for deep learning
# models (M4). Including them here:
#   (a) reveals artefacts (clipping, recording noise, room resonances)
#   (b) shows the harmonic structure and formant trajectories
#   (c) is the standard input for future CNN-based audio-to-respiratory models
#
# Plot one mel spectrogram per task per subject, with F0 overlay.

for sid, tasks in audio_data.items():
    for task, audio in tasks.items():
        mel = librosa.feature.melspectrogram(
            y=audio, sr=SR_TARGET,
            n_fft=FRAME, hop_length=HOP, n_mels=64
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)

        fig, ax = plt.subplots(figsize=(12, 3))
        img = librosa.display.specshow(
            mel_db, sr=SR_TARGET, hop_length=HOP,
            x_axis='time', y_axis='mel', ax=ax,
            fmin=50, fmax=SR_TARGET // 2
        )
        # Overlay F0 if available
        if sid in f0_data and task in f0_data[sid]:
            f0 = f0_data[sid][task]
            t_f0 = np.arange(len(f0)) * HOP / SR_TARGET
            ax.plot(t_f0, f0, color='yellow', lw=1.0, alpha=0.8, label='F0 (pYIN)')
            ax.legend(fontsize=8)
        plt.colorbar(img, ax=ax, format='%+2.0f dB')
        ax.set_title(f'{sid} — {task} | Mel spectrogram + F0')
        plt.tight_layout()
        plt.show()

## 13. Voce Girata vs Non-Girata — Paired Comparison

In [ ]:
# RELEVANCE: This is the **primary experimental contrast** of the study.
# Voce girata (covered, supported) vs non-girata (straight) on the same
# musical material (Scarborough Fair strofa) from the same singer.
#
# Expected acoustic differences:
#   - Girata: higher singer's formant ratio, shallower LTAS slope,
#             more regular vibrato, lower jitter, higher HNR,
#             lower F1 (vowel covering), higher centroid
#   - Non-girata: closer to speech norms; may show higher ZCR
#
# This paired comparison with 2 subjects is exploratory (insufficient for
# inference). It guides hypothesis formation for the full cohort.

GIRATA_LABEL     = 'scarb_girata'
NON_GIRATA_LABEL = 'scarb_non_girata'

comparison_rows = []
for subj in SUBJECTS:
    sid = subj['id']
    for label, condition in [(GIRATA_LABEL, 'girata'), (NON_GIRATA_LABEL, 'non_girata')]:
        if sid not in audio_data or label not in audio_data[sid]:
            continue
        audio = audio_data[sid][label]
        f0    = f0_data[sid].get(label, np.full(1, np.nan))

        ratio, ratio_db = compute_singers_formant_ratio(audio, SR_TARGET)
        slope, r2       = compute_ltas_slope(audio, SR_TARGET)
        rate, extent, reg = compute_vibrato(f0, F0_FRAME_RATE)
        hnr_f1, hnr_f2, _ = compute_formant_band_hnr(audio, SR_TARGET)
        rms_val = float(np.mean(librosa.feature.rms(y=audio, frame_length=FRAME, hop_length=HOP)))

        comparison_rows.append({
            'subject': sid, 'condition': condition,
            'sfr_db': ratio_db, 'ltas_slope': slope, 'ltas_r2': r2,
            'vib_rate': rate, 'vib_extent': extent, 'vib_regularity': reg,
            'hnr_f1': hnr_f1, 'hnr_f2': hnr_f2, 'rms': rms_val,
        })

df_cmp = pd.DataFrame(comparison_rows)

if not df_cmp.empty:
    print("\n=== Girata vs Non-Girata summary ===")
    display(df_cmp.groupby('condition').mean(numeric_only=True).T)

    feat_cols = ['sfr_db', 'ltas_slope', 'vib_rate', 'vib_extent',
                 'vib_regularity', 'hnr_f1', 'hnr_f2', 'rms']
    n = len(feat_cols)
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(16, 7))
    for ax, col in zip(axes.flat, feat_cols):
        sub = df_cmp.dropna(subset=[col])
        if sub.empty:
            ax.set_visible(False)
            continue
        sns.barplot(data=sub, x='condition', y=col, hue='subject', ax=ax)
        ax.set_title(col); ax.set_xlabel('')
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    fig.suptitle('Voce Girata vs Non-Girata — acoustic feature comparison', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No girata/non-girata tasks found — check task labels in audio_data.")

## 14. Scale Task Analysis — F0 Accuracy & Pitch Trajectory

In [ ]:
# RELEVANCE: Scale tasks (scale5_t1..t8, scale3, scale_asc_desc) are unique
# to the operatic protocol. They provide:
#   - Pitch accuracy: deviation of each note from its target semitone
#   - Intra-take consistency across the 8 semitone-shifted takes
#   - Transition dynamics: how quickly the singer moves between notes
#     (possibly correlated with respiratory flow events in OEP)
#
# This cell computes the median F0 per scale note and compares it to the
# equal-temperament target (A4=440 Hz reference).
#
# NOTE: automatic note segmentation would require onset detection or a
# score-alignment step — not implemented here. We visualise the full F0
# contour per take instead, which gives enough information for the
# pre-cohort validation.

A4_HZ = 440.0

def hz_to_midi(hz: np.ndarray) -> np.ndarray:
    with np.errstate(divide='ignore', invalid='ignore'):
        midi = 69 + 12 * np.log2(hz / A4_HZ)
    return np.where(np.isfinite(midi), midi, np.nan)

scale5_tasks = sorted([t for tasks in audio_data.values() for t in tasks if 'scale5' in t])
scale5_tasks = list(dict.fromkeys(scale5_tasks))  # deduplicate

if scale5_tasks:
    fig, axes = plt.subplots(len(scale5_tasks), 1,
                             figsize=(14, 2.2 * len(scale5_tasks)), sharex=False)
    if len(scale5_tasks) == 1:
        axes = [axes]
    for ax, task in zip(axes, scale5_tasks):
        for subj in SUBJECTS:
            sid = subj['id']
            if sid not in f0_data or task not in f0_data[sid]:
                continue
            f0 = f0_data[sid][task]
            midi = hz_to_midi(f0)
            t = np.arange(len(midi)) / F0_FRAME_RATE
            ax.plot(t, midi, lw=0.9, label=sid)
        ax.set_ylabel('MIDI note')
        ax.set_title(task, fontsize=9)
        ax.legend(fontsize=7)
        # Draw target MIDI lines for Mezzo-Soprano scale5_t1 (C5=72, D5=74, E5=76, F5=77, G5=79)
        for midi_tgt in [72, 74, 76, 77, 79, 77, 76, 74, 72]:
            ax.axhline(midi_tgt, ls=':', lw=0.5, color='gray', alpha=0.5)
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle('5-note scale takes — MIDI pitch trajectory (Mezzo-Soprano target lines shown)',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No scale5 tasks found yet.")

## 15. Feature Summary Table & Export

In [ ]:
# Consolidate all per-task scalar features into one DataFrame

def merge_on_subject_task(*dfs):
    out = dfs[0].copy()
    for df in dfs[1:]:
        out = out.merge(df, on=['subject', 'task'], how='outer', suffixes=('', '_dup'))
        dup_cols = [c for c in out.columns if c.endswith('_dup')]
        out = out.drop(columns=dup_cols)
    return out

dfs_to_merge = [df for df in [df_f0, df_vib, df_sfr, df_ltas, df_spec, df_fmt, df_hnr] if len(df) > 0]

if dfs_to_merge:
    df_all = merge_on_subject_task(*dfs_to_merge)
    # Append perturbation metrics (only reliable ones)
    df_pert_rel = df_pert.drop(columns=['reliable'], errors='ignore')
    df_all = df_all.merge(df_pert_rel, on=['subject', 'task'], how='outer')

    display(df_all)

    # Save to CSV
    out_dir = Path(r'C:\Users\Matéo\OneDrive\Documents\GitHub\pneumophonic_pipeline\data_target\singer_subjects\acoustic_features')
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / 'precohort_acoustic_features.csv'
    df_all.to_csv(out_path, index=False)
    print(f"Saved to {out_path}")
else:
    print("No data to consolidate — load audio files first.")

## 16. Feature Relevancy Summary

| Feature | Task types | OEP correlation potential | Priority |
|---|---|---|---|
| **F0 mean/median** | All | Moderate (pitch-flow coupling) | High |
| **Vibrato rate** | Sustained, Scarb, Aria | **High** — respiratory oscillation | High |
| **Vibrato extent** | Sustained, Scarb, Aria | **High** — flow amplitude | High |
| **Vibrato regularity** | Sustained, Scarb, Aria | Secondary (quality metric) | Medium |
| **Singer's formant ratio** | All | Low–moderate (projection strategy) | Medium |
| **LTAS slope** | All | Moderate (loudness → Vcw) | Medium |
| **RMS energy** | All | **High** — direct loudness proxy | High |
| **Spectral centroid** | All | Moderate (brightness ↔ subglottal P) | Medium |
| **HNR (band F1/F2)** | Sustained, Scarb | Moderate (glottal closure) | Medium |
| **Jitter / Shimmer** | Sustained only | Low (vibrato invalidates for singers) | Low |
| **F1/F2/F3** | Sustained, Scarb | Low (resonance strategy, not OEP) | Low |
| **MFCCs 1–13** | All | Moderate (spectral shape → M4 input) | Medium |
| **F0 range / SD** | Scales | Low (pitch task descriptor) | Low |

> **For the OEP correlation model (M3):** prioritise RMS, vibrato rate/extent, LTAS slope, and MFCCs 1–3.  
> **For the girata/non-girata contrast:** prioritise singer's formant ratio, HNR bands, vibrato regularity, LTAS slope.  
> **For scale tasks:** F0 accuracy and trajectory are descriptors; OEP correlation requires per-note respiratory segmentation (future work).